# SQ-MIL — Colab Bootstrap

Minimal, copy-paste setup for training SQ-MIL on a Colab A100.

**Design (three rules):**
1. **Code lives on Colab local disk** (`/content/SQ-MIL`), pulled from GitHub — fast, always latest.
2. **Data is copied Drive → local disk once per session** — training reads 513 small `.npy` per epoch; reading them straight off the Drive FUSE mount is ~10× slower.
3. **Checkpoints/results are written back to Drive** — Colab disconnects; weights must survive.

Edit-code loop: change code locally → `git push` → rerun **Cell 2** (`git pull`) here → retrain. Code flows one-way GitHub → Colab; data only ever lives in Drive.

Expected Drive layout (`MyDrive/SQ-MIL/`): `labels.csv`, `splits/splits_0..4.csv`, `ovarian_conch/{conch,sp_conch_n16_c50_512}/`, `results/`.

In [ ]:
# ── Cell 1: mount Drive + config ─────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# --- edit here if your paths differ ---
os.environ['DRIVE']    = '/content/drive/MyDrive/SQ-MIL'   # data root on Drive
os.environ['REPO_URL'] = 'https://github.com/z-pan/SQ-MIL.git'
os.environ['BRANCH']   = 'main'

assert os.path.isdir(os.environ['DRIVE']), f"Drive folder not found: {os.environ['DRIVE']}"

In [ ]:
# ── Cell 2: code → local disk (clone, or pull if present) + deps ──
%cd /content
!if [ -d SQ-MIL ]; then cd SQ-MIL && git fetch && git checkout "$BRANCH" && git pull; else git clone --branch "$BRANCH" "$REPO_URL"; fi
%cd /content/SQ-MIL
!pip install -q -r requirements.txt
!git log --oneline -1

In [ ]:
# ── Cell 3: data Drive → local disk (run once per session, ~2-4 min) ──
# rsync --ignore-existing makes reconnects near-instant.
!mkdir -p data/embeddings data/superpixels data/splits
!rsync -a --ignore-existing "$DRIVE/ovarian_conch/conch/"               data/embeddings/
!rsync -a --ignore-existing "$DRIVE/ovarian_conch/sp_conch_n16_c50_512/" data/superpixels/
!cp "$DRIVE/splits/"*.csv data/splits/
!cp "$DRIVE/labels.csv"   data/labels.csv

# sanity check: expect emb=513  sp=513  splits=5
!echo "emb=$(ls data/embeddings/*.npy | wc -l)  sp=$(ls data/superpixels/*.npy | wc -l)  splits=$(ls data/splits/*.csv | wc -l)"

In [ ]:
# ── Cell 4: train — reads local data/, writes checkpoints to Drive ──
# Stage 1 (all 5 folds). Checkpoints land in Drive, so a disconnect never loses a finished fold.
!bash scripts/04_train_stage1.sh \
    --config configs/ovarian_conch_s1.yaml \
    --all_folds \
    --output_dir "$DRIVE/results/stage1"

# Stage 2 (uncomment after Stage 1 finishes):
# !bash scripts/05_train_stage2.sh \
#     --config configs/ovarian_conch_s2.yaml \
#     --all_folds \
#     --output_dir "$DRIVE/results/stage2"

# Resume a single unfinished fold after a disconnect (rerun Cells 1-3 first):
# !bash scripts/04_train_stage1.sh --config configs/ovarian_conch_s1.yaml \
#     --fold 3 --output_dir "$DRIVE/results/stage1"